In [2]:
from pathlib import Path
import os

veri_koku = Path("/kaggle/input")

print("Bağlı veri setleri:")
for klasor in sorted(veri_koku.iterdir()):
    print("-", klasor.name)

print("\nKlasör yapısı ve örnek dosyalar:")
for sira, (klasor, alt_klasorler, dosyalar) in enumerate(
    os.walk(veri_koku)
):
    alt_klasorler.sort()
    dosyalar.sort()

    print(f"\n{Path(klasor).relative_to(veri_koku)}")
    print("  Alt klasörler:", alt_klasorler[:8])
    print(f"  Dosya sayısı: {len(dosyalar)}")
    print("  Örnek dosyalar:", dosyalar[:3])

    if sira >= 15:
        print("\nÖzet burada durduruldu.")
        break

Bağlı veri setleri:
- datasets

Klasör yapısı ve örnek dosyalar:

.
  Alt klasörler: ['datasets']
  Dosya sayısı: 0
  Örnek dosyalar: []

datasets
  Alt klasörler: ['emirhanuludogan']
  Dosya sayısı: 0
  Örnek dosyalar: []

datasets/emirhanuludogan
  Alt klasörler: ['paaaaa']
  Dosya sayısı: 0
  Örnek dosyalar: []

datasets/emirhanuludogan/paaaaa
  Alt klasörler: ['uavid_train', 'uavid_val']
  Dosya sayısı: 1
  Örnek dosyalar: ['readme.txt']

datasets/emirhanuludogan/paaaaa/uavid_train
  Alt klasörler: ['uavid_train']
  Dosya sayısı: 0
  Örnek dosyalar: []

datasets/emirhanuludogan/paaaaa/uavid_train/uavid_train
  Alt klasörler: ['seq1', 'seq10', 'seq11', 'seq12', 'seq13', 'seq14', 'seq15', 'seq2']
  Dosya sayısı: 0
  Örnek dosyalar: []

datasets/emirhanuludogan/paaaaa/uavid_train/uavid_train/seq1
  Alt klasörler: ['Images', 'Labels']
  Dosya sayısı: 0
  Örnek dosyalar: []

datasets/emirhanuludogan/paaaaa/uavid_train/uavid_train/seq1/Images
  Alt klasörler: []
  Dosya sayısı: 10
  Örne

In [3]:
from pathlib import Path
import pandas as pd

veri = Path("/kaggle/input/datasets/emirhanuludogan/paaaaa")
kayitlar = []

for bolum in ["uavid_train", "uavid_val"]:
    bolum_yolu = veri / bolum
    goruntu_klasorleri = sorted(
        p for p in bolum_yolu.rglob("Images") if p.is_dir()
    )
    if not goruntu_klasorleri:
        raise FileNotFoundError(f"Görüntü klasörü bulunamadı: {bolum}")

    for goruntu_klasoru in goruntu_klasorleri:
        sekans = goruntu_klasoru.parent
        etiket_klasoru = sekans / "Labels"

        goruntuler = {p.name for p in goruntu_klasoru.glob("*.png")}
        etiketler = {p.name for p in etiket_klasoru.glob("*.png")}

        kayitlar.append({
            "Bölüm": bolum,
            "Sekans": sekans.name,
            "Görüntü": len(goruntuler),
            "Etiket": len(etiketler),
            "Eşleşen": len(goruntuler & etiketler),
            "Etiketsiz": len(goruntuler - etiketler),
            "Görüntüsüz": len(etiketler - goruntuler),
            "Klasörler dolu": bool(goruntuler) and bool(etiketler),
        })

kontrol = pd.DataFrame(kayitlar)

print("SEKANS BAZINDA")
print(kontrol.to_string(index=False))

print("\nTOPLAMLAR")
print(
    kontrol.groupby("Bölüm")[
        ["Görüntü", "Etiket", "Eşleşen", "Etiketsiz", "Görüntüsüz"]
    ].sum().to_string()
)

print("\nSEKANS SAYILARI")
print(kontrol.groupby("Bölüm").size().to_string())

sorunlar = kontrol[
    (kontrol["Etiketsiz"] > 0)
    | (kontrol["Görüntüsüz"] > 0)
    | (~kontrol["Klasörler dolu"])
]
print("\nSorun bulunan sekans sayısı:", len(sorunlar))

SEKANS BAZINDA
      Bölüm Sekans  Görüntü  Etiket  Eşleşen  Etiketsiz  Görüntüsüz  Klasörler dolu
uavid_train   seq1       10      10       10          0           0            True
uavid_train  seq10       10      10       10          0           0            True
uavid_train  seq11       10      10       10          0           0            True
uavid_train  seq12       10      10       10          0           0            True
uavid_train  seq13       10      10       10          0           0            True
uavid_train  seq14       10      10       10          0           0            True
uavid_train  seq15       10      10       10          0           0            True
uavid_train   seq2       10      10       10          0           0            True
uavid_train   seq3       10      10       10          0           0            True
uavid_train  seq31       10      10       10          0           0            True
uavid_train  seq32       10      10       10          0      

In [4]:
from PIL import Image
from collections import Counter

boyutlar = Counter()
hatalar = []
kontrol_edilen = 0

for bolum in ["uavid_train", "uavid_val"]:
    klasorler = sorted(
        p for p in (veri / bolum).rglob("Images") if p.is_dir()
    )

    for klasor in klasorler:
        for goruntu_yolu in sorted(klasor.glob("*.png")):
            etiket_yolu = (
                klasor.parent / "Labels" / goruntu_yolu.name
            )

            try:
                with Image.open(goruntu_yolu) as goruntu:
                    goruntu.load()  # Sadece başlığı değil, pikselleri de oku.
                    goruntu_boyutu = goruntu.size

                with Image.open(etiket_yolu) as etiket:
                    etiket.load()
                    etiket_boyutu = etiket.size

                kontrol_edilen += 1
                boyutlar[
                    (bolum, goruntu_boyutu, etiket_boyutu)
                ] += 1

                if goruntu_boyutu != etiket_boyutu:
                    hatalar.append(
                        f"Boyut farkı: {goruntu_yolu.relative_to(veri)}"
                    )

            except Exception as hata:
                hatalar.append(
                    f"{goruntu_yolu.relative_to(veri)}: "
                    f"{type(hata).__name__}: {hata}"
                )

    print(f"{bolum} tamamlandı.", flush=True)

print("\nBaşarıyla okunan çift:", kontrol_edilen)
print("\nBoyut dağılımı — genişlik × yükseklik:")
for (bolum, goruntu_boyutu, etiket_boyutu), adet in sorted(
    boyutlar.items()
):
    print(
        f"{bolum}: görüntü={goruntu_boyutu}, "
        f"etiket={etiket_boyutu}, çift={adet}"
    )

print("\nHata sayısı:", len(hatalar))
for hata in hatalar[:20]:
    print("-", hata)

uavid_train tamamlandı.
uavid_val tamamlandı.

Başarıyla okunan çift: 270

Boyut dağılımı — genişlik × yükseklik:
uavid_train: görüntü=(3840, 2160), etiket=(3840, 2160), çift=130
uavid_train: görüntü=(4096, 2160), etiket=(4096, 2160), çift=70
uavid_val: görüntü=(3840, 2160), etiket=(3840, 2160), çift=40
uavid_val: görüntü=(4096, 2160), etiket=(4096, 2160), çift=30

Hata sayısı: 0


In [5]:
from pathlib import Path
from PIL import Image
from collections import Counter
import numpy as np
import pandas as pd

veri = Path("/kaggle/input/datasets/emirhanuludogan/paaaaa")

siniflar = {
    "Diğer / karma bölgeler": (0, 0, 0),
    "Bina": (128, 0, 0),
    "Yol": (128, 64, 128),
    "Ağaç": (0, 128, 0),
    "Alçak bitki örtüsü": (128, 128, 0),
    "Hareketli araç": (64, 0, 128),
    "Park hâlindeki araç": (192, 0, 192),
    "İnsan": (64, 64, 0),
}

renk_kodlari = {
    ad: r * 65536 + g * 256 + b
    for ad, (r, g, b) in siniflar.items()
}
bilinenler = set(renk_kodlari.values())
sonuclar = []

for bolum in ["uavid_train", "uavid_val"]:
    etiketler = sorted((veri / bolum).rglob("Labels/*.png"))
    if not etiketler:
        raise FileNotFoundError(f"Etiket bulunamadı: {bolum}")

    sayac = Counter()

    for sira, yol in enumerate(etiketler, start=1):
        with Image.open(yol) as kaynak:
            rgb = np.array(kaynak.convert("RGB"), dtype=np.uint32)

        # RGB rengini benzersiz bir tam sayıyla temsil et.
        kodlar = (
            rgb[:, :, 0] * 65536
            + rgb[:, :, 1] * 256
            + rgb[:, :, 2]
        )
        renkler, adetler = np.unique(kodlar, return_counts=True)
        sayac.update({
            int(renk): int(adet)
            for renk, adet in zip(renkler, adetler)
        })

        if sira % 20 == 0 or sira == len(etiketler):
            print(
                f"{bolum}: {sira}/{len(etiketler)} etiket okundu",
                flush=True,
            )

    toplam = sum(sayac.values())
    bilinmeyen = {
        kod: adet for kod, adet in sayac.items()
        if kod not in bilinenler
    }

    print(
        f"\n{bolum} — tanınmayan piksel: "
        f"{sum(bilinmeyen.values())}"
    )
    for kod, adet in list(bilinmeyen.items())[:10]:
        renk = (kod // 65536, (kod // 256) % 256, kod % 256)
        print("  Tanınmayan RGB:", renk, "| Piksel:", adet)

    for ad, kod in renk_kodlari.items():
        sonuclar.append({
            "Bölüm": bolum,
            "Sınıf": ad,
            "Piksel": sayac[kod],
            "Alan (%)": 100 * sayac[kod] / toplam,
        })

dagilim = pd.DataFrame(sonuclar)

print("\nSINIFLARIN ALAN PAYLARI (%)")
print(
    dagilim.pivot(
        index="Sınıf", columns="Bölüm", values="Alan (%)"
    ).round(3).to_string()
)

uavid_train: 20/200 etiket okundu
uavid_train: 40/200 etiket okundu
uavid_train: 60/200 etiket okundu
uavid_train: 80/200 etiket okundu
uavid_train: 100/200 etiket okundu
uavid_train: 120/200 etiket okundu
uavid_train: 140/200 etiket okundu
uavid_train: 160/200 etiket okundu
uavid_train: 180/200 etiket okundu
uavid_train: 200/200 etiket okundu

uavid_train — tanınmayan piksel: 0
uavid_val: 20/70 etiket okundu
uavid_val: 40/70 etiket okundu
uavid_val: 60/70 etiket okundu
uavid_val: 70/70 etiket okundu

uavid_val — tanınmayan piksel: 0

SINIFLARIN ALAN PAYLARI (%)
Bölüm                   uavid_train  uavid_val
Sınıf                                         
Alçak bitki örtüsü           13.724     14.154
Ağaç                         25.431     23.549
Bina                         28.365     33.976
Diğer / karma bölgeler       17.691     16.164
Hareketli araç                0.937      0.820
Park hâlindeki araç           1.156      0.985
Yol                          12.586     10.166
İnsan   

In [6]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

veri = Path("/kaggle/input/datasets/emirhanuludogan/paaaaa")
ornekler = []

for bolum in ["uavid_train", "uavid_val"]:
    klasorler = sorted(
        p for p in (veri / bolum).rglob("Images") if p.is_dir()
    )

    # Tekrarlanabilir seçim: sıralı listenin ilk ve son sekansı.
    for klasor in [klasorler[0], klasorler[-1]]:
        goruntu_yolu = sorted(klasor.glob("*.png"))[0]
        etiket_yolu = (
            klasor.parent / "Labels" / goruntu_yolu.name
        )
        ornekler.append((bolum, goruntu_yolu, etiket_yolu))

fig, eksenler = plt.subplots(
    len(ornekler), 3, figsize=(18, 4 * len(ornekler))
)

for satir, (bolum, goruntu_yolu, etiket_yolu) in enumerate(ornekler):
    with Image.open(goruntu_yolu) as kaynak:
        goruntu = kaynak.convert("RGB")
    with Image.open(etiket_yolu) as kaynak:
        etiket = kaynak.convert("RGB")

    # Yalnızca gösterim için küçült; kaynak dosyalar değişmez.
    genislik = 960
    boyut = (
        genislik,
        round(goruntu.height * genislik / goruntu.width),
    )
    goruntu = goruntu.resize(boyut, Image.Resampling.BILINEAR)
    etiket = etiket.resize(boyut, Image.Resampling.NEAREST)

    eksenler[satir, 0].imshow(goruntu)
    eksenler[satir, 1].imshow(etiket, interpolation="nearest")
    eksenler[satir, 2].imshow(goruntu)
    eksenler[satir, 2].imshow(
        etiket, alpha=0.4, interpolation="nearest"
    )

    kimlik = f"{bolum} / {goruntu_yolu.parent.parent.name}"
    for sutun, baslik in enumerate(
        ["Fotoğraf", "Referans etiket", "Üst üste görünüm"]
    ):
        eksenler[satir, sutun].set_title(f"{kimlik}\n{baslik}")
        eksenler[satir, sutun].axis("off")

plt.tight_layout()
plt.show()

In [7]:
import torch
from importlib.metadata import version, PackageNotFoundError

for paket in ["torch", "torchvision"]:
    try:
        print(f"{paket} sürümü:", version(paket))
    except PackageNotFoundError:
        print(f"{paket}: kurulu değil")

print("GPU kullanılabilir mi?", torch.cuda.is_available())
print("GPU sayısı:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    gpu = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {gpu.name} | "
        f"Bellek: {gpu.total_memory / 1024**3:.1f} GB"
    )

torch sürümü: 2.11.0+cu130
torchvision sürümü: 0.26.0+cu130
GPU kullanılabilir mi? True
GPU sayısı: 2
GPU 0: Tesla T4 | Bellek: 14.6 GB
GPU 1: Tesla T4 | Bellek: 14.6 GB


In [8]:
from pathlib import Path
from PIL import Image
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

VERI = Path("/kaggle/input/datasets/emirhanuludogan/paaaaa")
BOYUT = (512, 320)  # genişlik, yükseklik
YOK_SAY = 255

# Sıra, model çıktısındaki sınıf numaralarını belirler.
SINIFLAR = [
    ("Diğer / karma bölgeler", (0, 0, 0)),
    ("Bina", (128, 0, 0)),
    ("Yol", (128, 64, 128)),
    ("Ağaç", (0, 128, 0)),
    ("Alçak bitki örtüsü", (128, 128, 0)),
    ("Hareketli araç", (64, 0, 128)),
    ("Park hâlindeki araç", (192, 0, 192)),
    ("İnsan", (64, 64, 0)),
]

class UAVidDataset(Dataset):
    def __init__(self, bolum):
        self.ciftler = []
        for klasor in sorted((VERI / bolum).rglob("Images")):
            if not klasor.is_dir():
                continue
            for goruntu in sorted(klasor.glob("*.png")):
                etiket = klasor.parent / "Labels" / goruntu.name
                if not etiket.is_file():
                    raise FileNotFoundError(etiket)
                self.ciftler.append((goruntu, etiket))
        if not self.ciftler:
            raise ValueError(f"Veri bulunamadı: {bolum}")

    def __len__(self):
        return len(self.ciftler)

    def __getitem__(self, indeks):
        goruntu_yolu, etiket_yolu = self.ciftler[indeks]
        with Image.open(goruntu_yolu) as kaynak:
            goruntu = kaynak.convert("RGB")
        with Image.open(etiket_yolu) as kaynak:
            etiket = kaynak.convert("RGB")

        assert goruntu.size == etiket.size

        # Aynı ölçek ve konum her ikisine de uygulanır.
        hedef_w, hedef_h = BOYUT
        oran = min(hedef_w / goruntu.width, hedef_h / goruntu.height)
        yeni = (
            round(goruntu.width * oran),
            round(goruntu.height * oran),
        )
        goruntu = goruntu.resize(yeni, Image.Resampling.BILINEAR)
        etiket = etiket.resize(yeni, Image.Resampling.NEAREST)

        rgb = np.array(etiket)
        maske = np.full((yeni[1], yeni[0]), YOK_SAY, dtype=np.int64)
        for sinif_no, (_, renk) in enumerate(SINIFLAR):
            maske[np.all(rgb == renk, axis=2)] = sinif_no

        if np.any(maske == YOK_SAY):
            raise ValueError(f"Tanınmayan etiket rengi: {etiket_yolu}")

        sol = (hedef_w - yeni[0]) // 2
        ust = (hedef_h - yeni[1]) // 2

        tuval = np.zeros((hedef_h, hedef_w, 3), dtype=np.uint8)
        hedef = np.full((hedef_h, hedef_w), YOK_SAY, dtype=np.int64)
        tuval[ust:ust+yeni[1], sol:sol+yeni[0]] = np.array(goruntu)
        hedef[ust:ust+yeni[1], sol:sol+yeni[0]] = maske

        x = torch.from_numpy(tuval.copy()).permute(2, 0, 1).float() / 255
        y = torch.from_numpy(hedef)
        return x, y

egitim_verisi = UAVidDataset("uavid_train")
dogrulama_verisi = UAVidDataset("uavid_val")

egitim_yukleyici = DataLoader(
    egitim_verisi, batch_size=4, shuffle=True, num_workers=0
)
dogrulama_yukleyici = DataLoader(
    dogrulama_verisi, batch_size=4, shuffle=False, num_workers=0
)

x, y = next(iter(egitim_yukleyici))
print("Eğitim / doğrulama:", len(egitim_verisi), len(dogrulama_verisi))
print("Görüntü paketi:", tuple(x.shape))
print("Etiket paketi:", tuple(y.shape))
degerler = torch.unique(y).tolist()
print("Bu paketteki etiket değerleri:", degerler)

gecersiz = [
    deger for deger in degerler
    if deger not in list(range(len(SINIFLAR))) + [YOK_SAY]
]
print("Geçersiz sınıf değerleri:", gecersiz)

print("Görüntü veri tipi:", x.dtype)
print("Etiket veri tipi:", y.dtype)
print("Görüntü değer aralığı:", x.min().item(), x.max().item())

Eğitim / doğrulama: 200 70
Görüntü paketi: (4, 3, 320, 512)
Etiket paketi: (4, 320, 512)
Bu paketteki etiket değerleri: [0, 1, 2, 3, 4, 5, 6, 7, 255]
Geçersiz sınıf değerleri: []
Görüntü veri tipi: torch.float32
Etiket veri tipi: torch.int64
Görüntü değer aralığı: 0.0 1.0


In [9]:
import torch
from torch import nn

torch.manual_seed(42)
cihaz = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Görüntüyü küçülterek özellik çıkarır, sonra sınıf haritasına dönüştürür.
class BaslangicModeli(nn.Module):
    def __init__(self, sinif_sayisi):
        super().__init__()
        self.ag = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2),
            nn.ReLU(),
            nn.Conv2d(16, sinif_sayisi, kernel_size=1),
        )

    def forward(self, x):
        return self.ag(x)

model = BaslangicModeli(len(SINIFLAR)).to(cihaz)
kayip_fonksiyonu = nn.CrossEntropyLoss(ignore_index=YOK_SAY)

model.eval()
with torch.no_grad():
    skorlar = model(x.to(cihaz))
    kayip = kayip_fonksiyonu(skorlar, y.to(cihaz))

assert skorlar.shape == (
    x.shape[0], len(SINIFLAR), y.shape[1], y.shape[2]
)
assert torch.isfinite(kayip).item(), "Kayıp değeri sonlu değil."

print("Kullanılan cihaz:", cihaz)
print("Model çıktısı:", tuple(skorlar.shape))
print("Parametre sayısı:", sum(p.numel() for p in model.parameters()))
print("Eğitim öncesi kontrol kaybı:", round(kayip.item(), 4))
print("İleri geçiş kontrolü tamamlandı. Henüz eğitim yapılmadı.")

Kullanılan cihaz: cuda:0
Model çıktısı: (4, 8, 320, 512)
Parametre sayısı: 34008
Eğitim öncesi kontrol kaybı: 2.0729
İleri geçiş kontrolü tamamlandı. Henüz eğitim yapılmadı.


In [10]:
import time
from pathlib import Path
import torch
import pandas as pd

EPOCH_SAYISI = 3
sinif_sayisi = len(SINIFLAR)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
kayitlar = []
en_iyi_miou = -1.0

kayit_yolu = Path("/kaggle/working/baslangic_modeli_en_iyi.pt")

def bir_tur(yukleyici, egitim):
    model.train(egitim)
    toplam_kayip = 0.0
    toplam_piksel = 0
    karisiklik = torch.zeros(
        (sinif_sayisi, sinif_sayisi),
        dtype=torch.int64,
        device=cihaz,
    )

    for adim, (goruntuler, etiketler) in enumerate(yukleyici, 1):
        goruntuler = goruntuler.to(cihaz)
        etiketler = etiketler.to(cihaz)
        gecerli = etiketler != YOK_SAY
        piksel_sayisi = int(gecerli.sum().item())

        if piksel_sayisi == 0:
            raise ValueError("Paketin tamamı yok sayılan piksellerden oluşuyor.")

        with torch.set_grad_enabled(egitim):
            if egitim:
                optimizer.zero_grad(set_to_none=True)

            skorlar = model(goruntuler)
            kayip = kayip_fonksiyonu(skorlar, etiketler)

            if not torch.isfinite(kayip).item():
                raise RuntimeError("Kayıp sonlu değil; eğitim durduruldu.")

            if egitim:
                kayip.backward()
                optimizer.step()

        toplam_kayip += kayip.item() * piksel_sayisi
        toplam_piksel += piksel_sayisi

        if not egitim:
            tahmin = skorlar.argmax(dim=1)
            # Satırlar gerçek sınıf, sütunlar tahmin edilen sınıftır.
            kod = (
                etiketler[gecerli] * sinif_sayisi
                + tahmin[gecerli]
            )
            karisiklik += torch.bincount(
                kod, minlength=sinif_sayisi**2
            ).reshape(sinif_sayisi, sinif_sayisi)

        if adim % 10 == 0 or adim == len(yukleyici):
            asama = "Eğitim" if egitim else "Doğrulama"
            print(f"  {asama}: {adim}/{len(yukleyici)} paket", flush=True)

    return toplam_kayip / toplam_piksel, karisiklik


for epoch in range(1, EPOCH_SAYISI + 1):
    baslangic = time.perf_counter()
    print(f"\nEPOCH {epoch}/{EPOCH_SAYISI}", flush=True)

    egitim_kaybi, _ = bir_tur(egitim_yukleyici, egitim=True)
    dogrulama_kaybi, matris = bir_tur(
        dogrulama_yukleyici, egitim=False
    )

    matris = matris.double()
    dogru = matris.diag()
    birlesim = matris.sum(1) + matris.sum(0) - dogru

    iou = torch.full_like(birlesim, float("nan"))
    mevcut = birlesim > 0
    iou[mevcut] = dogru[mevcut] / birlesim[mevcut]
    miou = iou[mevcut].mean().item()

    kayitlar.append({
        "Epoch": epoch,
        "Eğitim kaybı": egitim_kaybi,
        "Doğrulama kaybı": dogrulama_kaybi,
        "Doğrulama mIoU": miou,
    })

    if miou > en_iyi_miou:
        en_iyi_miou = miou
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "validation_miou": miou,
            "siniflar": SINIFLAR,
            "boyut": BOYUT,
            "ignore_index": YOK_SAY,
            "model_adi": "BaslangicModeli",
        }, kayit_yolu)

    print(
        f"Eğitim kaybı: {egitim_kaybi:.4f} | "
        f"Doğrulama kaybı: {dogrulama_kaybi:.4f} | "
        f"mIoU: %{100 * miou:.2f} | "
        f"Süre: {time.perf_counter() - baslangic:.0f} saniye",
        flush=True,
    )

    print("Bu turun doğrulama IoU değerleri:")
    for (ad, _), deger in zip(SINIFLAR, iou.cpu().tolist()):
        metin = "hesaplanamadı" if pd.isna(deger) else f"%{100*deger:.2f}"
        print(f"  {ad}: {metin}")

gecmis = pd.DataFrame(kayitlar)
gecmis.to_csv("/kaggle/working/ilk_egitim_gecmisi.csv", index=False)

print("\nEĞİTİM ÖZETİ")
print(gecmis.to_string(index=False))
print("\nEn iyi model kaydı:", kayit_yolu)


EPOCH 1/3
  Eğitim: 10/50 paket
  Eğitim: 20/50 paket
  Eğitim: 30/50 paket
  Eğitim: 40/50 paket
  Eğitim: 50/50 paket
  Doğrulama: 10/18 paket
  Doğrulama: 18/18 paket
Eğitim kaybı: 1.9672 | Doğrulama kaybı: 1.6941 | mIoU: %4.69 | Süre: 127 saniye
Bu turun doğrulama IoU değerleri:
  Diğer / karma bölgeler: %0.00
  Bina: %33.49
  Yol: %4.04
  Ağaç: %0.00
  Alçak bitki örtüsü: %0.00
  Hareketli araç: %0.00
  Park hâlindeki araç: %0.00
  İnsan: %0.00

EPOCH 2/3
  Eğitim: 10/50 paket
  Eğitim: 20/50 paket
  Eğitim: 30/50 paket
  Eğitim: 40/50 paket
  Eğitim: 50/50 paket
  Doğrulama: 10/18 paket
  Doğrulama: 18/18 paket
Eğitim kaybı: 1.6781 | Doğrulama kaybı: 1.6289 | mIoU: %4.29 | Süre: 129 saniye
Bu turun doğrulama IoU değerleri:
  Diğer / karma bölgeler: %0.00
  Bina: %34.18
  Yol: %0.00
  Ağaç: %0.11
  Alçak bitki örtüsü: %0.00
  Hareketli araç: %0.00
  Park hâlindeki araç: %0.00
  İnsan: %0.00

EPOCH 3/3
  Eğitim: 10/50 paket
  Eğitim: 20/50 paket
  Eğitim: 30/50 paket
  Eğitim: 40/